# Databases with SQLAlchemy

**SQLAlchemy** is Python's most popular database toolkit. It provides:
- **ORM** (Object-Relational Mapper) — map Python classes to database tables
- **Core** — SQL Expression Language for lower-level SQL

Install: `pip install sqlalchemy`

We'll use **SQLite** in-memory for all examples — no server needed.

In [ ]:
import sqlalchemy
print(f'SQLAlchemy version: {sqlalchemy.__version__}')

## 1. Setup: Engine and Base

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, Session

# Create engine — 'sqlite:///:memory:' = in-memory SQLite
engine = create_engine('sqlite:///:memory:', echo=False)  # echo=True shows SQL

# DeclarativeBase — all models inherit from this
class Base(DeclarativeBase):
    pass

print('Engine and Base ready')

## 2. Defining Models

In [ ]:
from sqlalchemy import Column, Integer, String, Float, Boolean, DateTime, ForeignKey, Text
from sqlalchemy.orm import relationship
from datetime import datetime

class User(Base):
    __tablename__ = 'users'
    
    id = Column(Integer, primary_key=True)
    name = Column(String(100), nullable=False)
    email = Column(String(255), unique=True, nullable=False)
    is_active = Column(Boolean, default=True)
    created_at = Column(DateTime, default=datetime.now)
    
    # One-to-many relationship
    posts = relationship('Post', back_populates='author', cascade='all, delete-orphan')
    
    def __repr__(self):
        return f'User(id={self.id}, name={self.name!r}, email={self.email!r})'


class Post(Base):
    __tablename__ = 'posts'
    
    id = Column(Integer, primary_key=True)
    title = Column(String(200), nullable=False)
    content = Column(Text)
    published = Column(Boolean, default=False)
    created_at = Column(DateTime, default=datetime.now)
    
    author_id = Column(Integer, ForeignKey('users.id'), nullable=False)
    author = relationship('User', back_populates='posts')
    
    def __repr__(self):
        return f'Post(id={self.id}, title={self.title!r})'


# Create all tables
Base.metadata.create_all(engine)
print('Tables created:', list(Base.metadata.tables.keys()))

## 3. CRUD: Create

In [ ]:
from sqlalchemy.orm import Session

with Session(engine) as session:
    # Create users
    alice = User(name='Alice Smith', email='alice@example.com')
    bob = User(name='Bob Jones', email='bob@example.com')
    charlie = User(name='Charlie Brown', email='charlie@example.com')
    
    session.add_all([alice, bob, charlie])
    session.flush()  # assign IDs without committing
    
    # Create posts
    posts = [
        Post(title='Python Tips', content='Some Python tips...', published=True, author=alice),
        Post(title='Flask Guide', content='How to use Flask...', published=True, author=alice),
        Post(title='Draft Post', content='Work in progress...', published=False, author=alice),
        Post(title='SQLAlchemy Intro', content='Database with Python...', published=True, author=bob),
    ]
    session.add_all(posts)
    session.commit()
    
    print(f'Created {session.query(User).count()} users')
    print(f'Created {session.query(Post).count()} posts')

## 4. CRUD: Read — Querying

In [ ]:
from sqlalchemy import select, and_, or_, desc

with Session(engine) as session:
    # Get all users
    users = session.scalars(select(User)).all()
    print('All users:')
    for u in users:
        print(f'  {u}')
    
    # Get by primary key
    user = session.get(User, 1)
    print(f'\nUser #1: {user}')
    
    # Filter
    alice = session.scalars(select(User).where(User.email == 'alice@example.com')).first()
    print(f"\nFound Alice: {alice}")
    
    # All published posts
    published = session.scalars(
        select(Post).where(Post.published == True).order_by(desc(Post.created_at))
    ).all()
    print(f'\nPublished posts ({len(published)}):')
    for p in published:
        print(f'  [{p.author.name}] {p.title}')

## 5. CRUD: Update and Delete

In [ ]:
from sqlalchemy import update, delete

with Session(engine) as session:
    # Update one record
    user = session.get(User, 1)
    user.name = 'Alice Johnson'  # just change attribute
    session.commit()
    print(f'Updated name: {session.get(User, 1).name}')
    
    # Bulk update
    session.execute(
        update(Post).where(Post.author_id == 1, Post.published == False).values(published=True)
    )
    session.commit()
    
    all_alice_posts = session.scalars(select(Post).where(Post.author_id == 1)).all()
    print(f'\nAlice posts (all published now):')
    for p in all_alice_posts:
        print(f'  {p.title}: published={p.published}')
    
    # Delete a record
    post_to_delete = session.scalars(select(Post).where(Post.title == 'Draft Post')).first()
    if post_to_delete:
        session.delete(post_to_delete)
        session.commit()
        print(f'\nDeleted: Draft Post')
    
    print(f'Remaining posts: {session.query(Post).count()}')

## 6. Relationships and Joins

In [ ]:
from sqlalchemy import join

with Session(engine) as session:
    # Access related objects via relationship
    alice = session.get(User, 1)
    print(f"Alice's posts:")
    for post in alice.posts:
        print(f'  - {post.title}')
    
    # Explicit join query
    results = session.execute(
        select(User.name, Post.title)
        .join(Post, User.id == Post.author_id)
        .order_by(User.name)
    ).all()
    
    print('\nAll user-post pairs (JOIN):')
    for user_name, post_title in results:
        print(f'  {user_name:15} → {post_title}')

## Mini-Project: Blog Database

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Text, Boolean, DateTime, ForeignKey, func
from sqlalchemy.orm import DeclarativeBase, Session, relationship
from datetime import datetime

engine2 = create_engine('sqlite:///:memory:')

class Base2(DeclarativeBase):
    pass

class Author(Base2):
    __tablename__ = 'authors'
    id = Column(Integer, primary_key=True)
    name = Column(String(100), nullable=False)
    bio = Column(Text)
    articles = relationship('Article', back_populates='author')
    
    def __repr__(self): return f'Author({self.name!r})'

class Tag(Base2):
    __tablename__ = 'tags'
    id = Column(Integer, primary_key=True)
    name = Column(String(50), unique=True)

class Article(Base2):
    __tablename__ = 'articles'
    id = Column(Integer, primary_key=True)
    title = Column(String(200), nullable=False)
    content = Column(Text)
    views = Column(Integer, default=0)
    published = Column(Boolean, default=False)
    created_at = Column(DateTime, default=datetime.now)
    author_id = Column(Integer, ForeignKey('authors.id'))
    author = relationship('Author', back_populates='articles')

Base2.metadata.create_all(engine2)

# Seed data
with Session(engine2) as s:
    authors = [
        Author(name='Alice', bio='Python expert'),
        Author(name='Bob', bio='Web developer'),
    ]
    s.add_all(authors)
    s.flush()
    
    articles = [
        Article(title='Python Tips', views=1250, published=True, author=authors[0]),
        Article(title='Flask Deep Dive', views=890, published=True, author=authors[0]),
        Article(title='SQL Joins Explained', views=2100, published=True, author=authors[1]),
        Article(title='Draft Article', views=0, published=False, author=authors[1]),
    ]
    s.add_all(articles)
    s.commit()

# Stats report
from sqlalchemy import select, func, desc

with Session(engine2) as s:
    print('=== Blog Stats ===')
    
    # Top articles by views
    top = s.scalars(
        select(Article).where(Article.published == True).order_by(desc(Article.views)).limit(3)
    ).all()
    print('\nTop articles:')
    for a in top:
        print(f'  {a.views:5} views — {a.title} (by {a.author.name})')
    
    # Author stats
    stats = s.execute(
        select(Author.name, func.count(Article.id).label('total'),
               func.sum(Article.views).label('total_views'))
        .join(Article, Author.id == Article.author_id)
        .group_by(Author.id)
    ).all()
    
    print('\nAuthor stats:')
    for name, total, views in stats:
        print(f'  {name}: {total} articles, {views:,} total views')